# 4. Example warm-start / fine-tuning (optional)
**Before you start:** select the **Python (DECODE Intel)** kernel. Run each cell from top to bottom (Shift+Enter). If you change `config.yaml`, rerun the first cell to reload it. Nothing is saved unless a cell tells you it will save.
**Goal:** start a short example training run using the supplied weights. The new run learns from simulated example data; it is not automatically adapted to your microscope.

This is more advanced than notebook 3. Run the example prediction first if you mainly want to see DECODE work.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


## A. Check the engine and load example parameters
The helper initializes from the example weights with strict architecture matching, then starts a fresh optimizer. A checkpoint is different: it includes training state for resuming an interrupted run.

In [ ]:
import decode.utils.param_io
import torch
import inspect
import importlib
try:
    trainer = importlib.import_module("decode.neuralfitter.train.live_engine")  # 0.10.2
except ModuleNotFoundError as exc:
    if exc.name != "decode.neuralfitter.train.live_engine": raise
    trainer = importlib.import_module("decode.neuralfitter.train.train")  # development checkout
print('Training module:', inspect.getfile(trainer))
print('setup_trainer signature:', inspect.signature(trainer.setup_trainer))
param = decode.utils.param_io.load_params(str(project_path(CFG['paths']['example_parameters'])))
param.InOut.calibration_file = str(project_path(CFG['paths']['example_calibration']).resolve())
param.InOut.checkpoint_init = None
train_cfg = CFG['example_training']
param.Hardware.device = ('cuda:0' if torch.cuda.is_available() else 'cpu') if train_cfg['device'] == 'auto' else train_cfg['device']
param.Hardware.device_simulation = param.Hardware.device
param.Hardware.num_worker_train = train_cfg['workers']
param.HyperParameter.batch_size = train_cfg['batch_size']
param.HyperParameter.epochs = train_cfg['epochs']
param.HyperParameter.pseudo_ds_size = train_cfg['training_samples']
param.TestSet.test_size = train_cfg['test_samples']
# StepLR avoids conflicting scheduler defaults when DECODE loads the example YAML.
param.HyperParameter.learning_rate_scheduler = 'StepLR'
param.HyperParameter.learning_rate_scheduler_param = decode.utils.param_io.RecursiveNamespace(
    step_size=train_cfg['scheduler_step_size'], gamma=train_cfg['scheduler_gamma'])
param.HyperParameter.opt_param.lr = train_cfg['warmstart_learning_rate']  # illustrative; requires tuning
print('Example weights:', EXAMPLE / 'model.pt')


## B. Create the config and print the command
Set `example_training.write_config: true` in `config.yaml` and rerun from the first cell. This writes a config but does not start training.

In [ ]:
WRITE_CONFIG = CFG['example_training']['write_config']
if WRITE_CONFIG:
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / 'warmstart_runs').mkdir(exist_ok=True)
    param.InOut.experiment_out = str((WORK / 'warmstart_runs').resolve())
    config_path = WORK / 'warmstart_params.yaml'
    if config_path.exists(): raise FileExistsError(f'Choose a fresh path: {config_path}')
    decode.utils.param_io.save_params(str(config_path), param)
    helper = project_path(CFG['paths']['warmstart_helper'])
    print('In a terminal with the DECODE environment activated, run:')
    weights = project_path(CFG["paths"]["example_model"])
    print(f'python "{helper}" --param "{config_path}" --weights "{weights}"')
else:
    print('Set example_training.write_config: true in config.yaml to create the config and display the command.')


## C. Run the printed command in Terminal
Activate `decode_intel`, paste the command, and look for **WARM START** in the output. The helper loads the weights because DECODE 0.10.2’s standard training path does not use the `model_init` YAML field. It creates a new run; it does not overwrite the supplied model.

Two epochs are a smoke test. Fine-tuning is not guaranteed to be faster or more accurate; to assess it, compare against scratch training and validate on suitable data. This helper uses internal DECODE APIs and needs a real training test on your installation.
